# Aufgabe 2 – Hodgkin-Huxley-Gleichungssystem

Euler & RK4 (selbst), Vergleich mit scipy.odeint, Stromvariation, Stromimpuls.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt


## 2a) Euler-Verfahren, 50 ms, konstanter Strom $I=I_0 

In [ ]:
from src import hodgkin_huxley as hh

t = np.arange(0, 50, 0.01)
f = lambda y, t: hh.rhs(y, t, I_ext=-5.0)
y = hh.solve_euler(f, hh.initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, Euler, I₀ = −5 nA (Ruhezustand)")

## 2b) RK4 – Implementierung

In [ ]:


t = np.arange(0, 50, 0.01)
f = lambda y, t: hh.rhs(y, t, I_ext=-5.0)
y = hh.solve_rk4(f, hh.initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, RK4, I₀ = −5 nA (Ruhezustand)")

## 2b) Stabilitätsvergleich - Euler vs. RK4

In [ ]:

I0 = 10.0                          # spikeauslösender Strom -> steile Flanken fordern den Solver
dts = [0.01, 0.05, 0.08, 0.09]

fig, (axE, axR) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for dt in dts:
    t = np.arange(0, 50, dt)
    f = lambda y, t: hh.rhs(y, t, I0)
    UE = hh.solve_euler(f, hh.initial_state(), t)[:, 0]
    UR = hh.solve_rk4(f,   hh.initial_state(), t)[:, 0]
    axE.plot(t, UE, label=f"dt={dt}")
    axR.plot(t, UR, label=f"dt={dt}")

for ax, titel in ((axE, "Euler"), (axR, "RK4")):
    ax.set_title(titel); ax.set_xlabel("t [ms]"); ax.legend()
    ax.set_ylim(-100, 120)         # begrenzen, sonst zerdrückt die explodierende Kurve alles
axE.set_ylabel("U [mV]")
plt.tight_layout()
plt.show()

## 2b) odeint - Implementierung

In [ ]:
from scipy.integrate import odeint

t = np.arange(0, 50, 0.01)
f = lambda y, t: hh.rhs(y, t, I_ext=-5.0)
y = odeint(f, hh.initial_state(), t)
U = y[:, 0]

plt.plot(t, U)
plt.xlabel("t [ms]"); plt.ylabel("U [mV]")
plt.title("HHM, odeint, I₀ = −5 nA (Ruhezustand)")

## 2b) Performance Vergleich

In [ ]:
import time

def zeit(fn, wiederholungen=10):
    t0 = time.perf_counter()
    for _ in range(wiederholungen):
        fn()
    return (time.perf_counter() - t0) / wiederholungen * 1000  # ms pro Durchlauf

verfahren = {
    "Euler":  lambda: hh.solve_euler(f, hh.initial_state(), t),
    "RK4":    lambda: hh.solve_rk4(f, hh.initial_state(), t),
    "odeint": lambda: odeint(hh.rhs, hh.initial_state(), t, args=(-5.0,)),
}

for name, fn in verfahren.items():
    print(f"{name:8s}: {zeit(fn):.3f} ms")

## 2c) Variation von $I_0$ zwischen -5 nA und 15 nA mit RK4

In [ ]:
I_0 = [-5.0, 0.0, 5.0, 10.0, 15.0]  # Stromstärken in nA



## 2d) Stromimpuls (t=10..11 ms, I_imp=50 nA): U, I, n, m, h

In [ ]:
# TODO